# Day 09 — Async Fundamentals

============================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - Event loop — the single thread that manages everything
  - async def + await — writing coroutines
  - asyncio.run() — the entry point for async programs
  - Sync generators (yield) — the prerequisite for async generators
  - async with — async context managers (open + close safely)
  - The two most common async mistakes

WHY THIS MATTERS FOR LLM ENGINEERING:
  A FastAPI service handling 100 concurrent users makes 100 concurrent
  LLM API calls. Each call takes 3-8 seconds. Without async, requests
  queue up and users wait. With async, all 100 calls happen simultaneously.
  That is why asyncio exists — not for speed, but for CONCURRENCY.


In [ ]:

import asyncio
import time
import logging
import httpx          # async HTTP client (like requests but async)
from pathlib import Path

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")




## SECTION 1: EVENT LOOP — THE CORE CONCEPT

═════════════════════════════════════════════════════════════════════════════
ANALOGY: Think of the event loop as a restaurant waiter.
SYNCHRONOUS (no event loop):
The waiter takes Order A to the kitchen, stands at the counter,
watches the food cook for 8 minutes, then carries it to Table A.
Then takes Order B to the kitchen... Table B waits 16 minutes.
ASYNCHRONOUS (event loop):
The waiter takes Order A to the kitchen, says "call me when ready",
then immediately takes Order B to the kitchen, then Order C...
When the kitchen calls "Order A ready!", the waiter picks it up.
All three tables get served in ~8 minutes total.
The key insight: asyncio does NOT use multiple threads.
One thread. One event loop. Cooperative scheduling.
When an async function hits an `await`, it YIELDS control back to the
event loop, which can then run another coroutine.
THIS IS WHY: blocking the event loop (e.g. time.sleep inside async)
blocks ALL concurrent requests, not just the current one.


## SECTION 2: SYNC GENERATORS — PREREQUISITE FOR ASYNC GENERATORS

═════════════════════════════════════════════════════════════════════════════
Before learning async generators, understand sync generators first.
A generator function uses `yield` to produce values ONE AT A TIME.
The caller receives each value immediately, without waiting for all.


In [ ]:

def generate_prompt_chunks(long_document: str, chunk_size: int = 200):
    """
    Split a long document into chunks using a sync generator.

    A generator function returns a generator OBJECT, not a value.
    Each call to next() runs the function until the next `yield`.
    When `yield` is reached, the value is sent to the caller AND
    the function PAUSES (its state is preserved).

    This is the foundation of streaming:
      - Each `yield` sends one piece of data immediately
      - The caller processes it while the generator is paused
      - Then the caller asks for the next piece

    Args:
        long_document: Text to split into chunks.
        chunk_size    : Maximum characters per chunk.

    Yields:
        One string chunk at a time.
    """

    start = 0
    total = len(long_document)

    while start < total:
        # Calculate end of this chunk (don't go past the document end)
        end   = min(start + chunk_size, total)
        chunk = long_document[start:end]

        # `yield` sends the chunk to the caller and pauses here
        # The next time next() is called, execution resumes from HERE
        yield chunk

        start = end  # move to next chunk



In [ ]:
def build_few_shot_prompt_from_generator(examples: list[dict], max_examples: int = 3):
    """
    Yield formatted few-shot example strings one at a time.

    This generator pattern is used in RAG pipelines where you retrieve
    context documents and yield them into a prompt as they arrive.

    Yields:
        Formatted example strings.
    """

    for i, example in enumerate(examples):
        if i >= max_examples:
            return   # `return` inside a generator raises StopIteration

        yield f"Input : {example.get('input', '')}\nOutput: {example.get('output', '')}"




## SECTION 3: async def + await — WRITING COROUTINES

═════════════════════════════════════════════════════════════════════════════


In [ ]:

# THE MOST COMMON MISTAKE 1: Forgetting await


## async def get_response(prompt):

result = fetch_llm_response(prompt)   # ← WRONG: returns a coroutine object!
result = await fetch_llm_response(prompt)  # ← RIGHT: runs the coroutine
Without `await`, `result` is a coroutine object, not the actual response.
Python 3.11+ shows a RuntimeWarning, but older versions are silent.


In [ ]:

# THE MOST COMMON MISTAKE 2: Blocking the event loop


## async def slow_handler():

time.sleep(3)      # ← WRONG: blocks the ENTIRE event loop for 3 seconds
await asyncio.sleep(3)  # ← RIGHT: yields control to the event loop


In [ ]:

async def simulate_llm_api_call(prompt: str, delay: float = 1.0) -> str:
    """
    Simulate an async LLM API call.

    This is the pattern for any I/O-bound operation in async code:
    1. Start the operation
    2. await — give control back to the event loop
    3. When the operation completes, the event loop resumes THIS coroutine
    4. Return the result

    Args:
        prompt: The prompt to send (simulated).
        delay : How long the "API call" takes in seconds.

    Returns:
        A mock LLM response string.
    """

    log.info(f"→ LLM call started: '{prompt[:40]}...' (will take {delay}s)")

    # asyncio.sleep() is the async version of time.sleep()
    # It yields control to the event loop for `delay` seconds
    # During this time, OTHER coroutines can run
    await asyncio.sleep(delay)

    response = f"[LLM Response] '{prompt[:30]}...'"
    log.info(f"← LLM call done : '{prompt[:40]}...'")
    return response




In [ ]:

def run_sync_sequential(queries: list[str]) -> list[str]:
    """
    Process queries SYNCHRONOUSLY (one after another).

    Total time = sum of all individual call times.
    3 queries × 1 second each = 3 seconds total.
    """

    results = []
    for query in queries:
        # time.sleep blocks — cannot do anything else during this time
        time.sleep(1.0)   # simulates a slow API call
        results.append(f"[Sync] Response to: {query}")

    return results



In [ ]:
async def run_async_sequential(queries: list[str]) -> list[str]:
    """
    Process queries ASYNCHRONOUSLY but one after another.

    Each `await` yields to the event loop, but since we process
    one at a time, total time is still ~3 seconds.

    This is CORRECT async code but not OPTIMAL.
    Day 10 shows how to run them in parallel with asyncio.gather().
    """

    results = []
    for query in queries:
        # await gives control back to the event loop during the 1-second wait
        # Other coroutines could run here (if there were any scheduled)
        result = await simulate_llm_api_call(query, delay=1.0)
        results.append(result)

    return results




## SECTION 4: async with — ASYNC CONTEXT MANAGERS

═════════════════════════════════════════════════════════════════════════════
`with` statement (sync, Day 04):  manages resources synchronously
`async with` statement:            manages resources asynchronously
The pattern is IDENTICAL — async with just adds the ability to
await the open() and close() operations.
Use async with for:
- httpx.AsyncClient (async HTTP requests)
- async database connections (psycopg async, Day 10)
- Any resource that needs async setup/teardown


In [ ]:

async def fetch_product_data_async(product_ids: list[int]) -> list[dict]:
    """
    Fetch product data from a (simulated) API using httpx.AsyncClient.

    httpx.AsyncClient is an async HTTP client — the async equivalent
    of the requests library (covered in Day 11).

    `async with httpx.AsyncClient() as client:`
     ↑ opens the client   ↑ the client object
    When the `async with` block exits, the client is automatically closed.
    This prevents connection leaks.

    Args:
        product_ids: List of product IDs to look up.

    Returns:
        List of product data dicts (simulated).
    """

    results = []

    # async with: opens the HTTP client asynchronously
    # The client is automatically closed when the block exits (even on error)
    async with httpx.AsyncClient(timeout=10.0) as client:
        for product_id in product_ids:
            # In real code: response = await client.get(f"/api/products/{product_id}")
            # Here: simulate the async call
            await asyncio.sleep(0.1)   # simulate network latency

            # Simulated response data matching the products.csv schema
            results.append({
                "product_id"  : product_id,
                "product_name": f"Product {product_id}",
                "price"       : product_id * 10.5,
                "in_stock"    : product_id % 3 != 0,   # every 3rd is out of stock
            })

    # Client is guaranteed closed here (async with handled it)
    log.info(f"Fetched {len(results)} products (client auto-closed)")
    return results




## SECTION 5: async generators — PREVIEW OF SSE STREAMING (Day 12)

═════════════════════════════════════════════════════════════════════════════
An async generator is a generator (uses yield) that is also async (uses await).
This is the EXACT mechanism used to stream LLM tokens in a FastAPI endpoint.
How Day 12 uses this:
async def stream_tokens(prompt) → EventSourceResponse:
async for token in llm_client.stream(prompt):   # async for!
yield {"data": token}                        # send to browser


In [ ]:

async def stream_llm_tokens(prompt: str, num_tokens: int = 5):
    """
    Async generator that yields LLM tokens one at a time.

    This simulates what happens when you call an LLM with streaming=True.
    Instead of waiting for the entire response, tokens arrive individually.

    Usage:
        async for token in stream_llm_tokens("Hello"):
            print(token, end="", flush=True)

    Yields:
        Individual token strings.
    """

    # Simulate the LLM "thinking" before the first token
    await asyncio.sleep(0.2)

    words = f"Response to {prompt}".split()

    for word in words[:num_tokens]:
        # Simulate the delay between tokens (LLMs generate ~30-60 tokens/second)
        await asyncio.sleep(0.1)   # 100ms between tokens

        # `yield` in an async generator sends the token to the caller
        # The caller uses `async for` to receive each token
        yield word + " "




## ENTRY POINT: asyncio.run()

═════════════════════════════════════════════════════════════════════════════
asyncio.run() is the ONLY correct way to start an async program:
1. Creates a new event loop
2. Runs the given coroutine until completion
3. Closes the event loop and cleans up
You can only call asyncio.run() from SYNCHRONOUS code (not from inside
another coroutine). Inside a coroutine, just use `await`.


In [ ]:

async def main():
    """Main async entry point for Day 09 demonstration."""

    print("=" * 60)
    print("DAY 09 — Async Fundamentals")
    print("=" * 60)

    # Sync generators
    print("\n[1] Sync Generator (yield)")
    doc = "The quick brown fox jumps over the lazy dog. " * 3
    for i, chunk in enumerate(generate_prompt_chunks(doc, chunk_size=50)):
        print(f"  Chunk {i}: '{chunk}'")

    print("\n  Few-shot generator:")
    examples = [
        {"input": "Where is my order?", "output": "Please share your order ID."},
        {"input": "I want a refund.",    "output": "I'll start the return process."},
        {"input": "What is on sale?",    "output": "Check our promotions page."},
    ]
    for ex in build_few_shot_prompt_from_generator(examples, max_examples=2):
        print(f"  {ex}")

    # Single async call
    print("\n[2] Single async LLM call")
    start = time.perf_counter()
    response = await simulate_llm_api_call("What products are in stock?", delay=0.5)
    elapsed = round((time.perf_counter() - start) * 1000)
    print(f"  Response: {response}")
    print(f"  Elapsed : {elapsed}ms")

    # Sync vs Async comparison (sequential vs sequential)
    print("\n[3] Sync sequential vs Async sequential (3 queries)")
    queries = ["Query 1", "Query 2", "Query 3"]

    start = time.perf_counter()
    sync_results = run_sync_sequential(queries)
    sync_time = round(time.perf_counter() - start, 2)

    start = time.perf_counter()
    async_results = await run_async_sequential(queries)
    async_time = round(time.perf_counter() - start, 2)

    print(f"  Sync sequential  : {sync_time}s")
    print(f"  Async sequential : {async_time}s")
    print(f"  (Note: Similar times because both are sequential.")
    print(f"   Day 10 shows asyncio.gather() for parallel execution.)")

    # async with
    print("\n[4] async with — fetching products")
    products = await fetch_product_data_async([2001, 2002, 2003])
    for p in products:
        stock = "in stock" if p["in_stock"] else "out of stock"
        print(f"  Product {p['product_id']}: ${p['price']:.2f} ({stock})")

    # async generator
    print("\n[5] Async generator — streaming tokens")
    print("  Streaming: ", end="", flush=True)
    async for token in stream_llm_tokens("Hello world"):
        print(token, end="", flush=True)
    print()   # newline after streaming


if __name__ == "__main__":
    # asyncio.run() starts the event loop and runs main() to completion
    asyncio.run(main())


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day09_async_fundamentals.py', run_name='__main__')
